# Capacity Ladder — is it the signal, or the network's size?

**The question.** Our model scores ~0.847 with the tree bits and ~0.815 without them. Is that lift coming from the **information in the bits**, or from the network being **big enough to exploit anything**? And since the last experiment concluded that *the memorisation lives in the network, not in the bits*, this is the direct test of that claim: shrink the network and see whether the memorisation finally stops.

**The method.** Shrink the TabResNet trunk over four levels — a **56× parameter reduction** — and run **both views at every level**. The x-only arm is essential: it shows whether a small model can still learn at all, so a drop in the x+tree arm can be attributed to capacity rather than to the shrink breaking training.

| arm | d | d_hidden | blocks | params (x+tree) | params (x only) | shrink |
|---|---|---|---|---|---|---|
| **full** (current) | 256 | 512 | 4 | 2,439,682 | 1,057,538 | 1× |
| medium | 64 | 128 | 2 | 379,906 | 34,370 | 6.4× |
| small | 16 | 32 | 1 | 87,730 | 1,346 | 27.8× |
| **tiny** | 8 | 16 | 0 | 43,314 | 122 | 56.3× |

The tiny arm has **no residual blocks at all** — a linear map of the bits into 8 dimensions and then out to the two classes. If that still scores ~0.847, the tree bits are carrying the result almost by themselves.

### Logging: quiet console, complete CSV on Drive
**The training itself is untouched.** Logging only reads the loss tensor and the logits that
training already computed, under `no_grad`, after the optimiser step. It consumes no randomness,
adds no forward pass, and never touches a parameter or a BatchNorm statistic. Cell 5b proves it:
it trains twice with the same seed, with and without logging, and checks every per-epoch number
is bit-identical.

The console shows **only the usual per-epoch lines**. Every mini-batch (loss, within-epoch running-mean loss, accuracy, AUC) goes into `fusion_credit_batches.csv` — about 35,000 rows per model, 560,000 in all. Results are written locally while training runs and **synced to your Google Drive after each arm**, so a dropped session costs at most one arm.

### Plots
`plot_training.py` provides the plotting functions, so they can be reused on any run later:
```python
plot_training_curves(run_dir)   # 2x2: loss and AUC, at batch and epoch resolution
plot_loss(run_dir)              # loss only
plot_auc(run_dir)               # AUC only
plot_capacity(run_dirs)         # test AUC against parameter count
```

**Everything else is frozen** and identical to the last three experiments: credit, OOB-honest hard bits, AdamW (lr 3e-4, batch 128), 400 epochs with best-validation selection, 2-member ensembles, dropout and L1 off. The learning rate is deliberately **not** retuned per arm, so size is the only variable.

**What to look for**
1. **x+tree stays flat as the model shrinks** → the lift is the signal, not the capacity.
2. **The gap between the two curves widens as capacity falls** → the bits do work the raw features cannot do at that size.
3. **ep99** (the epoch train AUC first reaches 0.99) **moves later in the small arms** → capacity really is the memorisation channel, confirming the last report's conclusion.

⏱ **~50 minutes on an A100.** Runtime → GPU → Run all.

In [ ]:
# 1 · GPU check
import torch
print('CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU -> Runtime > GPU')

In [ ]:
# 2 · mount Google Drive — every CSV, JSON and figure ends up here
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/TKCE/capacity_ladder'
os.makedirs(DRIVE, exist_ok=True)
print('results will be saved to:', DRIVE)

In [ ]:
# 3 · get the code
%cd /content
!git clone https://github.com/sushanedulloo/TKCE.git 2>/dev/null || echo 'already cloned'
%cd /content/TKCE
!git pull

In [ ]:
# 4 · install deps
!pip install -q openml catboost optuna

In [ ]:
# 4b · OPTIONAL — only if OpenML 504s: upload openml_cache_clean5.tar.gz (else press Cancel)
import os, glob, tarfile
dst = '/root/.cache/openml/org/openml/www'
if glob.glob(dst + '/tasks/361055'):
    print('credit already cached — skip')
else:
    try:
        from google.colab import files; files.upload()
    except Exception as e: print('skipped:', e)
    hits = glob.glob('/content/**/openml_cache_clean5.tar.gz', recursive=True)
    if hits:
        os.makedirs(dst, exist_ok=True)
        with tarfile.open(hits[0]) as t: t.extractall(dst)
        print('cache extracted')
    else: print('no bundle — will use OpenML directly')

In [ ]:
# 5 · shared recipe + the Drive sync helper
#     --log-batches records EVERY batch to the CSV; the console still prints
#     only the per-epoch lines (add --log-batch-every 10 if you want batch lines too)
import shutil, os
base = ('--task 361055 --views "x;x+tree" --encoding oob --ensemble 2 --epochs 400 '
        '--dropout 0 --l1 0 --weight-decay 1e-3 --lr 3e-4 --batch-size 128 --device auto '
        '--log-batches')

def sync(arm):
    """Copy one arm's outputs to Drive. Training writes to local disk (fast and
    reliable); Drive gets the finished files, so a dropped session costs one arm."""
    src, dst = f'results/fusion/{arm}', f'{DRIVE}/{arm}'
    if not os.path.isdir(src):
        print(f'  !! {src} missing'); return
    shutil.copytree(src, dst, dirs_exist_ok=True)
    for f in sorted(os.listdir(dst)):
        print(f'  {os.path.getsize(os.path.join(dst, f))/1e6:8.2f} MB  {arm}/{f}')

print(base)
print('\n88 batches per epoch -> ~35,000 batch rows per model in the CSV')

In [ ]:
# 5b · PROOF that --log-batches does not change the training
#      Trains twice with the same seed, once plain and once with logging, and
#      checks every per-epoch number is bit-identical. ~1 minute.
!python -u verify_batch_logging.py --device auto --epochs 5

In [ ]:
# 6 · THE CAPACITY LADDER — each run trains x-only AND x+tree at that size
!python -u run_fusion.py {base} --d 256 --d-hidden 512 --n-blocks 4 --out results/fusion/cap_1_full
sync('cap_1_full')

In [ ]:
!python -u run_fusion.py {base} --d 64 --d-hidden 128 --n-blocks 2 --out results/fusion/cap_2_medium
sync('cap_2_medium')

In [ ]:
!python -u run_fusion.py {base} --d 16 --d-hidden 32 --n-blocks 1 --out results/fusion/cap_3_small
sync('cap_3_small')

In [ ]:
!python -u run_fusion.py {base} --d 8 --d-hidden 16 --n-blocks 0 --out results/fusion/cap_4_tiny
sync('cap_4_tiny')

In [ ]:
# 7 · summary table (also saved to Drive)
import json, glob, pandas as pd
rows = []
for d in sorted(glob.glob('results/fusion/cap_*')):
    js = [f for f in glob.glob(d + '/fusion_*.json')]
    if not js: continue
    s = json.load(open(js[0]))
    e = pd.read_csv(glob.glob(d + '/fusion_*_epochs.csv')[0])
    for r in s['results']:
        sub = e[e.model == r['model']].groupby('epoch').train_auc.mean()
        hit = sub[sub >= 0.99]
        rows.append(dict(arm=d.split('cap_')[-1], views=r['views'],
                         d=r['d'], blocks=r['n_blocks'], params=r['params'],
                         test_auc=r['test_auc'], best_val=r['best_val_auc'],
                         best_ep=r['best_epoch'], final_train=round(sub.iloc[-1], 4),
                         ep99=int(hit.index.min()) if len(hit) else None,
                         ceiling=s['tree_ceiling']))
t = pd.DataFrame(rows)
print(t.round(4).to_string(index=False))
print('\n--- the tree-bit lift at each size ---')
for arm in t.arm.unique():
    a = t[t.arm == arm]
    xo, xt = a[a.views == 'x'].test_auc, a[a.views == 'x+tree'].test_auc
    if len(xo) and len(xt):
        print(f'  {arm:10s} params={int(a[a.views=="x+tree"].params.iloc[0]):>9,d}  '
              f'x={xo.iloc[0]:.4f}  x+tree={xt.iloc[0]:.4f}  lift={xt.iloc[0]-xo.iloc[0]:+.4f}')
t.to_csv(f'{DRIVE}/cap_summary.csv', index=False)
print(f'\nsaved -> {DRIVE}/cap_summary.csv')

In [ ]:
# 7b · FINGERPRINT every arm — which arm, which data, which flags, which trajectory
#      Run this whenever a run 'looks different from last time'.
!python diagnose_run.py results/fusion/cap_*

In [ ]:
# 8 · plots — the reusable functions from plot_training.py
import glob
from plot_training import (plot_training_curves, plot_loss, plot_auc,
                           plot_capacity, plot_all)

# (a) the headline: test AUC against model size, both views
plot_capacity(sorted(glob.glob('results/fusion/cap_*')),
              out=f'{DRIVE}/cap_capacity_curve.png')

In [ ]:
# 8b · loss and AUC curves for every arm (batch resolution on the left,
#      epoch resolution on the right) — saved to Drive as well
for arm in ['cap_1_full', 'cap_2_medium', 'cap_3_small', 'cap_4_tiny']:
    plot_training_curves(f'results/fusion/{arm}',
                         out=f'{DRIVE}/{arm}_curves.png')

In [ ]:
# 8c · zoom: the first 20 epochs, where everything happens on this dataset
plot_loss('results/fusion/cap_1_full', max_epoch=20, out=f'{DRIVE}/full_loss_first20.png')
plot_auc('results/fusion/cap_1_full',  max_epoch=20, out=f'{DRIVE}/full_auc_first20.png')
plot_loss('results/fusion/cap_4_tiny', max_epoch=20, out=f'{DRIVE}/tiny_loss_first20.png')
plot_auc('results/fusion/cap_4_tiny',  max_epoch=20, out=f'{DRIVE}/tiny_auc_first20.png')

In [ ]:
# 9 · what is on Drive now
import os
tot = 0
for root, _, fs in os.walk(DRIVE):
    for f in sorted(fs):
        p = os.path.join(root, f); sz = os.path.getsize(p); tot += sz
        print(f'{sz/1e6:8.2f} MB  {os.path.relpath(p, DRIVE)}')
print(f'\ntotal: {tot/1e6:.1f} MB in {DRIVE}')